In [1]:
import pandas as pd
import numpy as np
from matplotlib import pyplot as plt
import os
import seaborn as sns
import sys
import glob
import tqdm
%unload_ext autotimebar

The autotimebar extension is not loaded.


In [2]:
cell_df = ("/extdata4/baeklab/Hyeonseo/m6A/inference/inference/AIRNA-DW-v2-20240827-175235-22-373000-token_normalise_dwell_all_npz-ON0090_allmotif_pileup/pileup.pkl")
cell_df = pd.read_pickle(cell_df)

In [3]:

cell_df = cell_df[cell_df["count_pm6a"] >= 20]

In [4]:
label_df = pd.read_csv("/extdata4/baeklab/Hyeonseo/m6A/runs/exp_MRNA/ON0090/ON0090/label/Baeklab.070.GP3.depth5_None.twm6astrict.tsv", sep="\t")
label_df = label_df[["id", "depth", "label", "m6A_level", "5mer", "drach"]]
label_df.rename(columns = {"id": "label_id","m6A_level": "dom_label"}, inplace = True)

In [5]:
exp_df = pd.read_csv("/extdata3/baeklab/Hyeonseo/m6A/res/m6asites/m6A_Jungmin_110823.tsv", sep="\t")
print(label_df)
print(exp_df)

                label_id  depth  label  dom_label   5mer  drach
0           NM_000084:34      7      0        0.0  GCAGA  False
1           NM_000084:36      7      0        0.0  AGAGA  False
2           NM_000084:38      7      0        0.0  AGAAT  False
3           NM_000084:39      7      0        0.0  GAATG  False
4           NM_000084:43      7      0        0.0  GCAGC  False
...                  ...    ...    ...        ...    ...    ...
30960573  NR_183446:1338     51     -3        0.0  AGAAG  False
30960574  NR_183446:1339     51     -3        0.0  GAAGC  False
30960575  NR_183446:1347     51     -3        0.0  TCATT  False
30960576  NR_183446:1350     51     -3        0.0  TTAGC  False
30960577  NR_183446:1353     51     -3        0.0  GCAGG  False

[30960578 rows x 6 columns]
          chr str       pos       DoM    SAC  GLORI  MICLIP2  M6ACE  \
0        chrX   +  47054489  0.288278  False   True    False  False   
1        chrX   +  47054489  0.288278  False   True    False 

In [7]:
exp_df["label_id"] = exp_df["NMID"] + ":" + exp_df["transcript_coordinate"].astype(str)
exp_df = exp_df[["label_id", "SAC","GLORI", "MICLIP2", "M6ACE"]]
exp_df = label_df.merge(exp_df, on = "label_id", how = "left")

In [8]:
print(exp_df)

                label_id  depth  label  dom_label   5mer  drach  SAC GLORI  \
0           NM_000084:34      7      0        0.0  GCAGA  False  NaN   NaN   
1           NM_000084:36      7      0        0.0  AGAGA  False  NaN   NaN   
2           NM_000084:38      7      0        0.0  AGAAT  False  NaN   NaN   
3           NM_000084:39      7      0        0.0  GAATG  False  NaN   NaN   
4           NM_000084:43      7      0        0.0  GCAGC  False  NaN   NaN   
...                  ...    ...    ...        ...    ...    ...  ...   ...   
30960573  NR_183446:1338     51     -3        0.0  AGAAG  False  NaN   NaN   
30960574  NR_183446:1339     51     -3        0.0  GAAGC  False  NaN   NaN   
30960575  NR_183446:1347     51     -3        0.0  TCATT  False  NaN   NaN   
30960576  NR_183446:1350     51     -3        0.0  TTAGC  False  NaN   NaN   
30960577  NR_183446:1353     51     -3        0.0  GCAGG  False  NaN   NaN   

         MICLIP2 M6ACE  
0            NaN   NaN  
1            

In [10]:
cell_exp_df = cell_df.merge(exp_df, on = "label_id", how = "inner")
cell_exp_df = cell_exp_df.fillna(0)
cell_exp_df[["SAC","GLORI", "MICLIP2", "M6ACE"]] = cell_exp_df[["SAC","GLORI", "MICLIP2", "M6ACE"]].astype(bool)
print(cell_exp_df)

                label_id      pm6a  count_pm6a       dom  count_dom  \
0         NM_000016:1000  0.000661       257.0  0.000000        258   
1         NM_000016:1008  0.000804       253.0  0.000000        255   
2         NM_000016:1009  0.002531       257.0  0.000000        258   
3         NM_000016:1010  0.003011       253.0  0.011628        258   
4         NM_000016:1014  0.003508       247.0  0.003968        252   
...                  ...       ...         ...       ...        ...   
15105476    NR_184306:85  0.011885        45.0  0.000000         45   
15105477    NR_184306:87  0.016118        38.0  0.065217         46   
15105478    NR_184306:88  0.014161        43.0  0.000000         45   
15105479    NR_184306:93  0.989961        34.0  0.674419         43   
15105480    NR_184306:98  0.026973        31.0  0.052632         38   

          mean_pred  depth  label  dom_label   5mer  drach    SAC  GLORI  \
0          0.001252    268      0        0.0  AAACT   True  False  Fals

In [12]:

cell_exp_df["threshold"] = 1-((0.9 ** (1-cell_exp_df["mean_pred"])) * (0.02 ** (cell_exp_df["mean_pred"])))
cell_exp_df.to_pickle("/extdata4/baeklab/Hyeonseo/m6A/inference/inference/AIRNA-DW-v2-20240827-175235-22-373000-token_normalise_dwell_all_npz-ON0090_allmotif_pileup/pileup_experiment.pkl")

In [13]:

fn_df =cell_exp_df[(cell_exp_df["dom_label"] > 0 ) & (cell_exp_df["pm6a"] < cell_exp_df["threshold"])]
print(fn_df)

                label_id      pm6a  count_pm6a       dom  count_dom  \
24        NM_000016:1070  0.026777       254.0  0.011450        262   
25        NM_000016:1071  0.232897       255.0  0.050000        260   
364          NM_000016:2  0.000849        50.0  0.000000         50   
533         NM_000016:49  0.005413       189.0  0.000000        193   
539          NM_000016:5  0.085874        51.0  0.019608         51   
...                  ...       ...         ...       ...        ...   
15101227   NR_182751:173  0.003858        53.0  0.035714         56   
15101880  NR_183060:1245  0.005252        30.0  0.031250         32   
15101883  NR_183060:1265  0.007946        30.0  0.000000         33   
15101926  NR_183060:1491  0.158222        31.0  0.060606         33   
15101999   NR_183060:450  0.001152        35.0  0.000000         36   

          mean_pred  depth  label  dom_label   5mer  drach    SAC  GLORI  \
24         0.015708    269     -1   0.033242  TGAAC  False  False   Tru

In [14]:
print(fn_df["SAC"].value_counts())
print(fn_df["GLORI"].value_counts())
print(fn_df["MICLIP2"].value_counts())
print(fn_df["M6ACE"].value_counts())

SAC
False    198956
True      20882
Name: count, dtype: int64
GLORI
True    219838
Name: count, dtype: int64
MICLIP2
False    192357
True      27481
Name: count, dtype: int64
M6ACE
False    205166
True      14672
Name: count, dtype: int64


In [18]:

fn_df[["SAC","GLORI", "MICLIP2", "M6ACE"]] = fn_df[["SAC","GLORI", "MICLIP2", "M6ACE"]].astype(int)
fn_df["support"] = fn_df["SAC"] + fn_df["GLORI"] + fn_df["MICLIP2"] + fn_df["M6ACE"]
print(fn_df["support"].value_counts())

support
1    168139
2     41163
3      9736
4       800
Name: count, dtype: int64


/tmp/ipykernel_152246/1030160500.py:1: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fn_df[["SAC","GLORI", "MICLIP2", "M6ACE"]] = fn_df[["SAC","GLORI", "MICLIP2", "M6ACE"]].astype(int)
/tmp/ipykernel_152246/1030160500.py:2: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  fn_df["support"] = fn_df["SAC"] + fn_df["GLORI"] + fn_df["MICLIP2"] + fn_df["M6ACE"]


In [19]:

cell_exp_df[["SAC","GLORI", "MICLIP2", "M6ACE"]] = cell_exp_df[["SAC","GLORI", "MICLIP2", "M6ACE"]].astype(int)
cell_exp_df["support"] = cell_exp_df["SAC"] + cell_exp_df["GLORI"] + cell_exp_df["MICLIP2"] + cell_exp_df["M6ACE"]
print(cell_exp_df["support"].value_counts())

support
0    14004553
1      906607
2      132503
3       51364
4       10454
Name: count, dtype: int64


In [20]:

tp_df =cell_exp_df[(cell_exp_df["dom_label"] > 0 ) & (cell_exp_df["pm6a"] >= cell_exp_df["threshold"])]
print(tp_df["support"].value_counts())

support
1    58152
2    51124
3    40232
4     9653
Name: count, dtype: int64


In [27]:

cell_exp_df["support2"] = cell_exp_df["GLORI"] + cell_exp_df["M6ACE"] + cell_exp_df["SAC"]
print(cell_exp_df["support2"].value_counts())

support2
0    14251387
1      746672
2       94189
3       13233
Name: count, dtype: int64


In [30]:

tp_df =cell_exp_df[(cell_exp_df["dom_label"] > 0.1 ) & (cell_exp_df["pm6a"] >= cell_exp_df["threshold"])]
print(tp_df["support"].value_counts())

support
1    48733
2    48034
3    39499
4     9585
Name: count, dtype: int64


In [31]:

fn_df =cell_exp_df[(cell_exp_df["dom_label"] > 0.1 ) & (cell_exp_df["pm6a"] < cell_exp_df["threshold"])]
print(fn_df["support"].value_counts())

support
1    18266
2     7956
3     3562
4      255
Name: count, dtype: int64
